In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path

split_dir = Path("/content/drive/MyDrive/NLP/splits")

train_df = pd.read_csv(split_dir / "train.csv", keep_default_na=False)
val_df = pd.read_csv(split_dir / "validation.csv", keep_default_na=False)
test_df = pd.read_csv(split_dir / "test.csv", keep_default_na=False)

X_train = train_df["review_comment"].to_numpy(dtype=str)
X_val = val_df["review_comment"].to_numpy(dtype=str)
X_test = test_df["review_comment"].to_numpy(dtype=str)

y_train = train_df["label"].to_numpy(dtype="int32")
y_val = val_df["label"].to_numpy(dtype="int32")
y_test = test_df["label"].to_numpy(dtype="int32")

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(42)

BATCH_SIZE = 64
MAX_TOKENS = 15000

def make_dataset(texts, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=len(texts),
            seed=42
        )

    return ds.batch(BATCH_SIZE).prefetch(1)

In [4]:
train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

In [5]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="tf_idf"
)

adapt_ds = tf.data.Dataset.from_tensor_slices(X_train).batch(BATCH_SIZE)
vectorizer.adapt(adapt_ds)

In [6]:
import numpy as np
from scipy.sparse import csr_matrix, vstack
from sklearn.linear_model import Ridge

def get_tfidf_features(texts):
    batches = tf.data.Dataset.from_tensor_slices(texts).batch(BATCH_SIZE)

    return vstack(
        [
            csr_matrix(vectorizer(batch).numpy())
            for batch in batches
        ],
        format="csr"
    )

X_train_tfidf = get_tfidf_features(X_train)
X_val_tfidf = get_tfidf_features(X_val)

model = Ridge(
    alpha=1.0,
    solver="lsqr"
)

In [7]:
model.fit(
    X_train_tfidf,
    y_train.astype("float32")
)

print("Ridge regression trained.")

Ridge regression trained.


In [8]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    mean_absolute_error,
    mean_squared_error
)

actual_scores = np.asarray(y_val).ravel() + 1
raw_predictions = model.predict(X_val_tfidf).ravel() + 1

predicted_scores = np.clip(raw_predictions, 1, 5)

print("Validation MAE:", mean_absolute_error(
    actual_scores, predicted_scores
))

print("Validation RMSE:", np.sqrt(mean_squared_error(
    actual_scores, predicted_scores
)))

predicted_classes = np.floor(predicted_scores + 0.5).astype(int)

print("\nClassification report after rounding:")
print(classification_report(
    actual_scores,
    predicted_classes,
    labels=[1, 2, 3, 4, 5],
    target_names=["Score 1", "Score 2", "Score 3", "Score 4", "Score 5"],
    zero_division=0
))

print("Rounded MAE:", mean_absolute_error(
    actual_scores, predicted_classes
))

Validation MAE: 0.795418918132782
Validation RMSE: 1.1608942426834659

Classification report after rounding:
              precision    recall  f1-score   support

     Score 1       0.67      0.42      0.51      1383
     Score 2       0.12      0.22      0.16       339
     Score 3       0.16      0.21      0.18       564
     Score 4       0.21      0.44      0.28       950
     Score 5       0.77      0.55      0.64      3266

    accuracy                           0.46      6502
   macro avg       0.39      0.37      0.36      6502
weighted avg       0.58      0.46      0.50      6502

Rounded MAE: 0.7794524761611812


In [9]:
# from pathlib import Path

# model_dir = Path("/content/drive/MyDrive/NLP/models")
# model_dir.mkdir(parents=True, exist_ok=True)

# model_path = model_dir / "review_score_model.keras"

# model.save(str(model_path))

# print("Model saved:", model_path)